# Chapter 6 — The Differential

Companion notebook to Chapter 6 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Two tools from `torch.func` are used throughout:

* `jacrev(f)(a)` returns the Jacobian matrix of `f` at `a`;
* `jvp(f, (a,), (v,))` returns `f(a)` and `Df(a) v`, the image of a velocity,
  without forming the matrix.

In [1]:
import math
import warnings
import torch
from torch.func import jacrev, jvp

warnings.filterwarnings("ignore", category=FutureWarning)

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)

2.15.0+cu132


## 1. The remainder is small compared with `h`

Example 6.6 and Figure 6.2: `f(rho, theta) = (rho cos theta, rho sin theta)`
at `a = (2, pi/6)`. Dividing `h` by 10 divides the remainder by 100.

In [2]:
def polar(p):
    return torch.stack([p[0] * torch.cos(p[1]), p[0] * torch.sin(p[1])])

a = torch.tensor([2.0, math.pi / 6])
J = jacrev(polar)(a)
print("Jacobian at a:\n", J)
assert torch.allclose(J, torch.tensor([[math.sqrt(3) / 2, -1.0],
                                       [0.5, math.sqrt(3)]]))

for s in (0.1, 0.01, 0.001):
    h = torch.tensor([s, s])
    exact = polar(a + h)
    tangent = polar(a) + J @ h
    r = (exact - tangent).norm().item()
    print(f"h = ({s}, {s}):  f(a+h) = {[round(v, 4) for v in exact.tolist()]}"
          f"  tangent = {[round(v, 4) for v in tangent.tolist()]}"
          f"  |r| = {r:.3e}  |r|/|h| = {r / h.norm().item():.3e}")

h = torch.tensor([0.1, 0.1])
assert torch.allclose(polar(a + h), torch.tensor([1.7047, 1.2263]), atol=5e-5)
assert torch.allclose(polar(a) + J @ h, torch.tensor([1.7187, 1.2232]), atol=5e-5)
r1 = (polar(a + h) - polar(a) - J @ h).norm().item()
r2 = (polar(a + h / 10) - polar(a) - J @ h / 10).norm().item()
assert abs(r1 - 0.0143) < 5e-5 and abs(r2 - 0.000142) < 5e-7
assert 95 < r1 / r2 < 105

Jacobian at a:
 tensor([[ 0.8660, -1.0000],
        [ 0.5000,  1.7321]])
h = (0.1, 0.1):  f(a+h) = [1.7047, 1.2263]  tangent = [1.7187, 1.2232]  |r| = 1.425e-02  |r|/|h| = 1.008e-01
h = (0.01, 0.01):  f(a+h) = [1.7306, 1.0224]  tangent = [1.7307, 1.0223]  |r| = 1.415e-04  |r|/|h| = 1.001e-02
h = (0.001, 0.001):  f(a+h) = [1.7319, 1.0022]  tangent = [1.7319, 1.0022]  |r| = 1.414e-06  |r|/|h| = 1.000e-03


## 2. The differential acts on velocities

Theorem 6.10: for any curve `c` through `a` with velocity `v`, the image
curve `f o c` has velocity `Df(a) v`. Two different curves with the same
velocity at `a` give the same answer.

In [3]:
v = torch.tensor([0.3, -0.7])

def line(t):
    return a + t * v

def bent(t):                       # same point and velocity at t = 0
    return a + t * v + t ** 2 * torch.tensor([5.0, 2.0]) + torch.sin(t) ** 3

t0 = torch.tensor(0.0)
vel_line = jvp(lambda t: polar(line(t)), (t0,), (torch.tensor(1.0),))[1]
vel_bent = jvp(lambda t: polar(bent(t)), (t0,), (torch.tensor(1.0),))[1]
print("velocity of f(line):", vel_line.tolist())
print("velocity of f(bent):", vel_bent.tolist())
print("Df(a) v            :", (J @ v).tolist())
assert torch.allclose(vel_line, J @ v) and torch.allclose(vel_bent, J @ v)
# the same thing in one call
assert torch.allclose(jvp(polar, (a,), (v,))[1], J @ v)

velocity of f(line): [0.9598076211353315, -1.0624355652982143]
velocity of f(bent): [0.9598076211353315, -1.0624355652982143]
Df(a) v            : [0.9598076211353315, -1.0624355652982143]


## 3. Directional derivatives are not enough

The pitfall of Section 6.3: `f(x, y) = x^3 y / (x^6 + y^2)` has all its
directional derivatives at the origin equal to 0, and is not continuous
there. Exercise 6.3: `x^2 y / (x^2 + y^2)` is continuous with directional
derivatives `f(v)`, not linear in `v`.

In [4]:
def f_bad(x, y):
    return x ** 3 * y / (x ** 6 + y ** 2)

for v1, v2 in ((1.0, 1.0), (1.0, -2.0), (3.0, 0.5)):
    q = [f_bad(t * v1, t * v2) / t for t in (1e-1, 1e-2, 1e-3, 1e-4)]
    print(f"v = ({v1}, {v2}): f(tv)/t =", [f"{x:.2e}" for x in q])
    assert abs(q[-1]) < 1e-2 and abs(q[-1] / q[-2] - 0.1) < 1e-3   # tends to 0 like t
print("along y = x^3:", [f_bad(x, x ** 3) for x in (1e-1, 1e-2, 1e-3)])
assert all(abs(f_bad(x, x ** 3) - 0.5) < 1e-12 for x in (1e-1, 1e-2, 1e-3))

def f_ex(x, y):
    return x ** 2 * y / (x ** 2 + y ** 2)

d = lambda v1, v2: f_ex(1e-6 * v1, 1e-6 * v2) / 1e-6
print("Exercise 6.3: D_(1,0) =", d(1, 0), " D_(0,1) =", d(0, 1), " D_(1,1) =", d(1, 1))
assert abs(d(1, 1) - 0.5) < 1e-9 and d(1, 0) == 0 and d(0, 1) == 0

v = (1.0, 1.0): f(tv)/t = ['1.00e-01', '1.00e-02', '1.00e-03', '1.00e-04']
v = (1.0, -2.0): f(tv)/t = ['-5.00e-02', '-5.00e-03', '-5.00e-04', '-5.00e-05']
v = (3.0, 0.5): f(tv)/t = ['4.18e+00', '5.40e-01', '5.40e-02', '5.40e-03']
along y = x^3: [0.5, 0.5000000000000001, 0.5]
Exercise 6.3: D_(1,0) = 0.0  D_(0,1) = 0.0  D_(1,1) = 0.5


## 4. The Jacobian matrix under a change of frames

Corollary 6.15: with bases changed by `P` (source) and `Q` (target), the
Jacobian matrix becomes `Q^{-1} J P`, and moving the origins changes nothing.
For a scalar function the row of partial derivatives becomes `[df] P`
(Proposition 6.19): partial derivatives are covector components.

In [5]:
P = torch.tensor([[2.0, 1.0], [0.5, 3.0]])
Q = torch.tensor([[1.0, -1.0], [2.0, 0.5]])
o_src = torch.tensor([0.3, -0.2])           # new origins
o_tgt = torch.tensor([5.0, 1.0])

def polar_new(u):                           # f in the new frames
    x = o_src + P @ u
    return torch.linalg.solve(Q, polar(x) - o_tgt)

u_a = torch.linalg.solve(P, a - o_src)      # new coordinates of the point a
J_new = jacrev(polar_new)(u_a)
print("Jacobian in the new frames:\n", J_new)
assert torch.allclose(J_new, torch.linalg.solve(Q, J @ P))

# the pitfall: f = x1 x2 with u1 = x1, u2 = x1 + x2
f = lambda x: x[0] * x[1]
x = torch.tensor([1.5, 4.0])
P2 = torch.tensor([[1.0, 0.0], [-1.0, 1.0]])    # x = P2 u
g = lambda u: f(P2 @ u)
u = torch.linalg.solve(P2, x)
row_x = jacrev(f)(x)
row_u = jacrev(g)(u)
print("[df] in x:", row_x.tolist(), "  [df] in u:", row_u.tolist())
assert torch.allclose(row_u, row_x @ P2)
assert torch.allclose(row_u, torch.tensor([x[1] - x[0], x[0]]))
print("df/du1 =", row_u[0].item(), " differs from df/dx1 =", row_x[0].item(),
      " although u1 = x1")

# the number df(a)(h) does not depend on the basis
h = torch.tensor([0.2, -0.1])
assert torch.allclose(row_x @ h, row_u @ torch.linalg.solve(P2, h))

Jacobian in the new frames:
 tensor([[ 0.9928,  1.8517],
        [-0.2392,  3.9856]])
[df] in x: [4.0, 1.5]   [df] in u: [2.5, 1.5]
df/du1 = 2.5  differs from df/dx1 = 4.0  although u1 = x1


## 5. Critical points do not depend on coordinates

Exercise 6.8: `f = x^2 + x y + y^2 - 3 x` has its critical point at `(2, -1)`,
which is `(u, v) = (1, -1)` in the coordinates `u = x + y`, `v = y`.

In [6]:
f8 = lambda p: p[0] ** 2 + p[0] * p[1] + p[1] ** 2 - 3 * p[0]
crit = torch.tensor([2.0, -1.0])
print("df at (2, -1):", (jacrev(f8)(crit) + 0.0).tolist())
assert torch.allclose(jacrev(f8)(crit), torch.zeros(2))
T = torch.tensor([[1.0, -1.0], [0.0, 1.0]])     # (x, y) = T (u, v)
g8 = lambda q: f8(T @ q)
q = torch.linalg.solve(T, crit)
print("(u, v) =", q.tolist(), " df there:", (jacrev(g8)(q) + 0.0).tolist())
assert torch.allclose(q, torch.tensor([1.0, -1.0]))
assert torch.allclose(jacrev(g8)(q), torch.zeros(2))

df at (2, -1): [0.0, 0.0]
(u, v) = [1.0, -1.0]  df there: [0.0, 0.0]


## 6. Differentials without coordinates

Section 6.7. Each formula is compared with `jvp`, which applies the
differential to an increment `H`.

In [7]:
n = 4
A = torch.randn(n, n) + 3 * torch.eye(n)
B = torch.randn(n, n)
H = torch.randn(n, n)
K = torch.randn(n, n)

# bilinear: the matrix product
_, d = jvp(lambda X, Y: X @ Y, (A, B), (H, K))
assert torch.allclose(d, H @ B + A @ K)

# the square: AH + HA, not 2AH
_, d = jvp(lambda X: X @ X, (A,), (H,))
assert torch.allclose(d, A @ H + H @ A)
print("|D(A^2)H - 2AH| =", (d - 2 * A @ H).norm().item(), "(not zero)")
assert (d - 2 * A @ H).norm() > 1e-3

# the cube and A^T A (Exercise 6.5)
_, d = jvp(lambda X: X @ X @ X, (A,), (H,))
assert torch.allclose(d, A @ A @ H + A @ H @ A + H @ A @ A)
_, d = jvp(lambda X: X.T @ X, (A,), (H,))
assert torch.allclose(d, H.T @ A + A.T @ H)
_, d = jvp(lambda X: torch.trace(X.T @ X), (A,), (H,))
assert torch.allclose(d, 2 * torch.trace(A.T @ H))

# the inverse
Ai = torch.linalg.inv(A)
_, d = jvp(torch.linalg.inv, (A,), (H,))
assert torch.allclose(d, -Ai @ H @ Ai)

# the determinant and its logarithm
_, d = jvp(torch.linalg.det, (A,), (H,))
assert torch.allclose(d, torch.linalg.det(A) * torch.trace(Ai @ H))
_, d = jvp(lambda X: torch.logdet(X), (A,), (H,))
assert torch.linalg.det(A) > 0 and torch.allclose(d, torch.trace(Ai @ H))
print("bilinear, square, cube, A^T A, inverse, det and log det: all agree")

|D(A^2)H - 2AH| = 10.59153155141321 (not zero)
bilinear, square, cube, A^T A, inverse, det and log det: all agree


In [8]:
# the remainder of the inverse is bounded by 2 |A^{-1}|^3 |H|^2  (proof of 6.26)
op = lambda X: torch.linalg.matrix_norm(X, ord=2).item()
radius = 1 / (2 * op(Ai))                 # the proof needs |H| <= radius
print("|A^{-1}| =", round(op(Ai), 4), "  radius =", round(radius, 4))
for s in (radius, radius / 10, radius / 100):
    Hs = s * H / op(H)
    assert op(Hs) <= radius * (1 + 1e-12)
    rem = torch.linalg.inv(A + Hs) - Ai + Ai @ Hs @ Ai
    bound = 2 * op(Ai) ** 3 * op(Hs) ** 2
    print(f"|H| = {s:.2e}: remainder {op(rem):.3e} <= bound {bound:.3e}")
    assert op(rem) <= bound

# Exercise 6.6
A2 = torch.tensor([[2.0, 1.0], [1.0, 1.0]])
H2 = torch.tensor([[0.0, 1.0], [0.0, 0.0]])
A2i = torch.linalg.inv(A2)
print("D inv(A) H =", (-A2i @ H2 @ A2i).round(decimals=12).tolist())
print("d det(A) H =", (torch.linalg.det(A2) * torch.trace(A2i @ H2)).item())
assert torch.allclose(-A2i @ H2 @ A2i, torch.tensor([[1.0, -2.0], [-1.0, 2.0]]))
assert abs((torch.linalg.det(A2) * torch.trace(A2i @ H2)).item() + 1) < 1e-12

|A^{-1}| = 17.281   radius = 0.0289
|H| = 2.89e-02: remainder 4.984e-01 <= bound 8.641e+00
|H| = 2.89e-03: remainder 5.796e-03 <= bound 8.641e-02
|H| = 2.89e-04: remainder 5.893e-05 <= bound 8.641e-04
D inv(A) H = [[1.0, -2.0], [-1.0, 2.0]]
d det(A) H = -1.0


### Remark 6.28: the transpose of "matrix calculus"

`autograd` stores the differential of `det` at `A` as an array of partial
derivatives with the shape of `A`. That array is `det(A) A^{-T}`: the
representative of the linear form `H -> det(A) tr(A^{-1} H)` for the inner
product `<X, Y> = tr(X^T Y)`.

In [9]:
Ag = A.clone().requires_grad_(True)
(grad,) = torch.autograd.grad(torch.linalg.det(Ag), Ag)
C = torch.linalg.det(A) * Ai
assert torch.allclose(grad, C.T)
assert torch.allclose((grad * H).sum(), torch.trace(C @ H))
print("autograd array equals det(A) A^{-T}; pairing with H gives",
      (grad * H).sum().item())

autograd array equals det(A) A^{-T}; pairing with H gives 27.46588588673267


## 7. Layers

The machine-learning box of Section 6.8: affine layer, activation, softmax
and cross-entropy.

In [10]:
m, n = 3, 5
W = torch.randn(m, n); b = torch.randn(m); x = torch.randn(n)
Hw = torch.randn(m, n); beta = torch.randn(m); xi = torch.randn(n)
layer = lambda W, b, x: W @ x + b

assert torch.allclose(jacrev(lambda x: layer(W, b, x))(x), W)
_, d = jvp(lambda W, b: layer(W, b, x), (W, b), (Hw, beta))
assert torch.allclose(d, Hw @ x + beta)
_, d = jvp(layer, (W, b, x), (Hw, beta, xi))
assert torch.allclose(d, Hw @ x + W @ xi + beta)

# elementwise activation: a diagonal matrix
assert torch.allclose(jacrev(torch.tanh)(x), torch.diag(1 - torch.tanh(x) ** 2))

# ReLU at 0: the library returns 0, a convention
z0 = torch.zeros(1, requires_grad=True)
(g0,) = torch.autograd.grad(torch.relu(z0).sum(), z0)
print("autograd 'derivative' of ReLU at 0:", g0.item())
print("one-sided quotients at 0: right", (torch.relu(torch.tensor(1e-8)) / 1e-8).item(),
      " left", (torch.relu(torch.tensor(-1e-8)) / -1e-8).item())
assert g0.item() == 0.0

autograd 'derivative' of ReLU at 0: 0.0
one-sided quotients at 0: right 1.0  left -0.0


In [11]:
# softmax and cross-entropy, with the numbers of Exercise 6.7
z = torch.tensor([0.0, 0.0, math.log(2.0)])
p = torch.softmax(z, 0)
Jp = jacrev(lambda z: torch.softmax(z, 0))(z)
print("p =", p.tolist())
print("16 * Dp(z) =\n", (16 * Jp).round(decimals=10))
assert torch.allclose(p, torch.tensor([0.25, 0.25, 0.5]))
assert torch.allclose(Jp, torch.diag(p) - torch.outer(p, p))
assert torch.allclose(16 * Jp, torch.tensor([[3.0, -1, -2], [-1, 3, -2], [-2, -2, 4]]))
ones = torch.ones(3)
assert torch.allclose(Jp @ ones, torch.zeros(3), atol=1e-15)   # kernel
assert torch.allclose(ones @ Jp, torch.zeros(3), atol=1e-15)   # image in sum = 0

y = 2                                                           # label 3
ce = lambda z: -z[y] + torch.logsumexp(z, 0)
row = jacrev(ce)(z)
print("[d ce(z)] =", row.tolist())
assert torch.allclose(row, p - torch.eye(3)[y])
assert abs(row.sum().item()) < 1e-15

p = [0.25, 0.25, 0.5]
16 * Dp(z) =
 tensor([[ 3., -1., -2.],
        [-1.,  3., -2.],
        [-2., -2.,  4.]])
[d ce(z)] = [0.25, 0.25, -0.5]


## 8. The linearized model

For a small network `f(theta, x)`, the tangent affine map at `theta0` is
`f(theta0, x) + D_theta f(theta0, x)(theta - theta0)`. The error is
`o(theta - theta0)`: here it decreases like the square of the distance.

In [12]:
torch.manual_seed(1)
sizes = [(8, 1), (8,), (1, 8), (1,)]
theta0 = torch.cat([0.8 * torch.randn(s).flatten() for s in sizes])

def net(theta, x):
    i = 0; parts = []
    for s in sizes:
        k = math.prod(s); parts.append(theta[i:i + k].reshape(s)); i += k
    W1, b1, W2, b2 = parts
    return (W2 @ torch.tanh(W1 @ x + b1) + b2).squeeze()

xs = torch.linspace(-2, 2, 9).reshape(-1, 1)
direction = torch.randn_like(theta0); direction /= direction.norm()

def lin(theta, x):
    out, d = jvp(lambda th: net(th, x), (theta0,), (theta - theta0,))
    return out + d

prev = None
for eps in (1.0, 0.1, 0.01, 0.001):
    theta = theta0 + eps * direction
    err = max(abs(net(theta, x) - lin(theta, x)).item() for x in xs)
    note = "" if prev is None else f"   ratio {prev / err:.1f}"
    print(f"|theta - theta0| = {eps:6.3f}   max error = {err:.3e}{note}")
    if prev is not None and eps < 0.1:
        assert 80 < prev / err < 125
    prev = err

# the features of the linearized model, and the kernel K(x, x') = J(x) J(x')^T
feats = torch.stack([jacrev(lambda th: net(th, x))(theta0) for x in xs])
Kmat = feats @ feats.T
print("feature matrix:", tuple(feats.shape), "  kernel matrix:", tuple(Kmat.shape))
assert torch.allclose(Kmat, Kmat.T) and torch.linalg.eigvalsh(Kmat)[0] > -1e-10

|theta - theta0| =  1.000   max error = 6.566e-01
|theta - theta0| =  0.100   max error = 6.997e-03   ratio 93.8
|theta - theta0| =  0.010   max error = 6.966e-05   ratio 100.4
|theta - theta0| =  0.001   max error = 6.962e-07   ratio 100.1
feature matrix: (9, 25)   kernel matrix: (9, 9)


In [13]:
# Exercise 6.9: f(w, x) = w2 tanh(w1 x)
f9 = lambda w, x: w[1] * torch.tanh(w[0] * x)
x = torch.tensor(0.7)
row = jacrev(f9)(torch.tensor([0.0, 1.0]), x)
print("row at w0 = (0, 1):", row.tolist(), " -> linearized model w1 * x")
assert torch.allclose(row, torch.tensor([0.7, 0.0]))
row = jacrev(f9)(torch.tensor([1.0, 1.0]), x)
assert torch.allclose(row, torch.stack([x * (1 - torch.tanh(x) ** 2), torch.tanh(x)]))
print("row at w0 = (1, 1):", [round(v, 6) for v in row.tolist()])

row at w0 = (0, 1): [0.7, 0.0]  -> linearized model w1 * x
row at w0 = (1, 1): [0.444318, 0.604368]


## 9. Checking a differential by finite differences

The quotient `(f(a + t v) - f(a)) / t` approximates `Df(a) v` with an error
proportional to `t`, until round-off takes over near `t = 1e-8`. The
symmetric quotient has an error proportional to `t^2`.

In [14]:
exact = J @ v
print("    t      forward error   symmetric error")
best_fwd, best_sym = float("inf"), float("inf")
for k in range(1, 13):
    t = 10.0 ** (-k)
    fwd = (polar(a + t * v) - polar(a)) / t
    sym = (polar(a + t * v) - polar(a - t * v)) / (2 * t)
    e1, e2 = (fwd - exact).norm().item(), (sym - exact).norm().item()
    best_fwd, best_sym = min(best_fwd, e1), min(best_sym, e2)
    print(f"  1e-{k:02d}     {e1:.2e}        {e2:.2e}")
print("best forward error:", f"{best_fwd:.1e}", "  best symmetric error:", f"{best_sym:.1e}")
assert 1e-10 < best_fwd < 1e-6 and best_sym < best_fwd / 10

    t      forward error   symmetric error
  1e-01     5.35e-02        1.36e-03
  1e-02     5.33e-03        1.36e-05
  1e-03     5.33e-04        1.36e-07
  1e-04     5.33e-05        1.36e-09
  1e-05     5.33e-06        1.18e-11
  1e-06     5.33e-07        1.01e-10
  1e-07     5.38e-08        3.38e-10
  1e-08     3.72e-08        1.02e-08
  1e-09     1.07e-07        6.76e-08
  1e-10     2.12e-07        2.12e-07
  1e-11     2.01e-05        1.58e-05
  1e-12     1.97e-04        1.50e-04
best forward error: 3.7e-08   best symmetric error: 1.2e-11


## Summary

* The remainder `f(a + h) - f(a) - Df(a) h` shrinks faster than `h`.
* `Df(a)` sends velocities to velocities, whatever curve carries them.
* The Jacobian matrix changes as `Q^{-1} J P`; the row of partial
  derivatives of a function changes as `[df] P`.
* Differentials of matrix maps are computed as linear maps of an increment
  `H`, and `jvp` checks them without any `n^2 x n^2` matrix.
* A network is close to its tangent affine map in parameter space, with an
  error of second order.